# 01. Профілювання сирих даних, аудит дублікатів та реконструкція обласних інтервалів (2022–2026)

Цей блокнот перевіряє первинний датасет `data/raw/official_data_uk.csv` (`299,275` рядків, SHA-256: `1f50085481d229f57b624d83b1ce723e884dea409c14afb5b7040c8a2f233e4f`, часовий діапазон `2022-03-15 16:10:34 UTC` – `2026-09-07 05:28:22 UTC`), демонструє наявність здвоєного блоку рядків `0..113,910` та `113,911..227,831` безпосередньо в upstream-репозиторії (`Vadimkin/ukrainian-air-raid-sirens-dataset`, коміт `58ee75bc20113181b9ddf8029d4cbf3a62d8cd10`), а також відтворює повний баланс видалених рядків та реконструкцію обласних інтервалів у `data/processed/alerts_clean_oblast.parquet` (`84,908` інтервалів).

In [1]:
import hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

raw_path = Path("../data/raw/official_data_uk.csv")
clean_path = Path("../data/processed/alerts_clean_oblast.parquet")

sha256 = hashlib.sha256(raw_path.read_bytes()).hexdigest()
df_raw = pd.read_csv(raw_path)
print(f"SHA-256: {sha256}")
print(f"Кількість сирих рядків: {len(df_raw):,}")
print(f"Рівні адмін. одиниць (сирі):\n{df_raw['level'].value_counts()}")

SHA-256: 1f50085481d229f57b624d83b1ce723e884dea409c14afb5b7040c8a2f233e4f
Кількість сирих рядків: 299,275
Рівні адмін. одиниць (сирі):
level
raion      132251
oblast     130235
hromada     36789
Name: count, dtype: int64


## 1. Верифікація дублювання блоків `0..113,910` та `113,911..227,831` в upstream CSV
В upstream-файлі `datasets/official_data_uk.csv` блок рядків `0..113,910` (`2022-03-15` – `2026-01-25`) повторюється у рядках `113,911..227,831` (`113,845` точних дублікатів + `76` ретроспективно оновлених записів). Перевіримо структуру трьох блоків і кількість дублікатів до та після очищення:

In [2]:
key_cols = ["oblast", "level", "raion", "hromada", "started_at", "finished_at"]
b1 = df_raw.iloc[0:113911]
b2 = df_raw.iloc[113911:227832]
b3 = df_raw.iloc[227832:]

print(f"Блок 1 (0..113,910):       {len(b1):,} рядків | {b1['started_at'].min()} -> {b1['started_at'].max()}")
print(f"Блок 2 (113,911..227,831): {len(b2):,} рядків | {b2['started_at'].min()} -> {b2['started_at'].max()}")
print(f"Блок 3 (227,832..299,274): {len(b3):,} рядків | {b3['started_at'].min()} -> {b3['started_at'].max()}")

raw_dups = int(df_raw.duplicated(subset=key_cols).sum())
df_dedup = df_raw.drop_duplicates(subset=key_cols)
print(f"\nТочних дублікатів у сирому CSV: {raw_dups:,}")
print(f"Унікальних рядків після дедуплікації: {len(df_dedup):,}")
print(f"Рівні адмін. одиниць після дедуплікації:\n{df_dedup['level'].value_counts()}")

Блок 1 (0..113,910):       113,911 рядків | 2022-03-15 16:10:34+00:00 -> 2026-01-25 13:11:52+00:00
Блок 2 (113,911..227,831): 113,921 рядків | 2022-03-15 16:10:34+00:00 -> 2026-01-25 13:11:52+00:00
Блок 3 (227,832..299,274): 71,443 рядків | 2026-01-23 12:42:58+00:00 -> 2026-09-07 05:28:22+00:00



Точних дублікатів у сирому CSV: 113,845
Унікальних рядків після дедуплікації: 185,430
Рівні адмін. одиниць після дедуплікації:
level
raion      99304
oblast     65314
hromada    20812
Name: count, dtype: int64


## 2. Перехід логування з `oblast` на `raion` / `hromada` у серпні–листопаді 2025 р.
Починаючи з серпня–листопада 2025 року, для 23 областей (усіх, крім м. Київ) записи `level == 'oblast'` майже зникли (22–113 на місяць замість ~1,500–3,000), оскільки тривоги стали фіксуватися на рівні районів (`raion`) та громад (`hromada`).

In [3]:
df_dedup = df_dedup.copy()
df_dedup["started_at_dt"] = pd.to_datetime(df_dedup["started_at"], utc=True)
df_dedup["ym"] = df_dedup["started_at_dt"].dt.strftime("%Y-%m")
monthly_levels = df_dedup[df_dedup["ym"] >= "2025-05"].groupby(["ym", "level"]).size().unstack(fill_value=0)
display(monthly_levels)

level,hromada,oblast,raion
ym,,,
2025-05,596,1352,317
2025-06,634,1463,340
2025-07,693,1766,596
2025-08,631,1117,2437
2025-09,572,705,3468
2025-10,693,715,3498
2025-11,708,164,7266
2025-12,692,24,7298
2026-01,704,58,7665


## 3. Перевірка очищеного датасету `alerts_clean_oblast.parquet` (`assert 0` дублікатів)
З `2025-08-01` у `src/data/cleaner.py` обласна тривога реконструюється як об'єднання перетинних інтервалів `[started_at, finished_at]` усіх рівнів (`oblast`, `raion`, `hromada`) у межах кожної області.

In [4]:
df_clean = pd.read_parquet(clean_path)
clean_dups = int(df_clean.duplicated(subset=["oblast", "started_at", "finished_at"]).sum())
assert clean_dups == 0, f"Found {clean_dups} duplicates in cleaned dataset!"

print(f"Очищених обласних інтервалів: {len(df_clean):,} (дублікатів після очищення: {clean_dups})")
print(f"Діапазон дат: {df_clean['started_at'].min()} — {df_clean['started_at'].max()}")
print(f"Медіанна тривалість: {df_clean['duration_minutes'].median():.2f} хв, середня: {df_clean['duration_minutes'].mean():.2f} хв")
display(df_clean.head())

Очищених обласних інтервалів: 84,908 (дублікатів після очищення: 0)
Діапазон дат: 2022-03-15 16:10:34+00:00 — 2026-09-07 04:38:21+00:00
Медіанна тривалість: 43.15 хв, середня: 102.46 хв


,oblast,raion,hromada,level,started_at,finished_at,source,is_finished,duration_seconds,duration_minutes,...,started_at_kyiv,finished_at_kyiv,start_year,start_month,start_date,start_hour_kyiv,start_day_of_week,start_day_num,is_night,split_set
0,Вінницька область,None,None,oblast,2022-03-15 16:10:34+00:00,2022-03-15 16:50:07+00:00,official,True,2373.0,39.550000,...,2022-03-15 18:10:34+02:00,2022-03-15 18:50:07+02:00,2022,3,2022-03-15,18,Tuesday,1,False,train
1,Житомирська область,None,None,oblast,2022-03-15 16:11:25+00:00,2022-03-15 16:54:23+00:00,official,True,2578.0,42.966667,...,2022-03-15 18:11:25+02:00,2022-03-15 18:54:23+02:00,2022,3,2022-03-15,18,Tuesday,1,False,train
2,Кіровоградська область,None,None,oblast,2022-03-15 16:15:11+00:00,2022-03-15 16:54:52+00:00,official,True,2381.0,39.683333,...,2022-03-15 18:15:11+02:00,2022-03-15 18:54:52+02:00,2022,3,2022-03-15,18,Tuesday,1,False,train
3,Івано-Франківська область,None,None,oblast,2022-03-15 16:17:28+00:00,2022-03-15 17:14:18+00:00,official,True,3410.0,56.833333,...,2022-03-15 18:17:28+02:00,2022-03-15 19:14:18+02:00,2022,3,2022-03-15,18,Tuesday,1,False,train
4,Львівська область,None,None,oblast,2022-03-15 16:17:29+00:00,2022-03-15 17:16:23+00:00,official,True,3534.0,58.900000,...,2022-03-15 18:17:29+02:00,2022-03-15 19:16:23+02:00,2022,3,2022-03-15,18,Tuesday,1,False,train
